# m47 lab · Data and deployment engineering: units, versions, labels, privacy, quantization and model cards

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You write ECGs to WFDB files and catch header mistakes with an audit; fingerprint a dataset so that a silent change
shows up; combine labels from several labellers; measure re-identification risk in a synthetic registry and
coarsen it; shrink a model to 8 and 4 bits; and write a model card that records all of it. Part G audits real
PhysioNet headers and metadata.

| Part | What you do | Data |
| --- | --- | --- |
| A | Units in WFDB headers, and an audit that catches header mistakes (eq-m47-adu) | synthetic ECGs |
| B | Checksums, a manifest and a dataset version id | synthetic files |
| C | Several labellers, majority votes and agreement (eq-m47-majority) | synthetic labels |
| D | Re-identification risk and coarsening (eq-m47-risk) | synthetic registry |
| E | Quantizing inputs and weights (eq-m47-affine, eq-m47-qnoise) | synthetic beats |
| F | A model card generated from the lab's own results | parts B and E |
| G | Real headers, label-quality fields and an amplitude audit across databases | PTB-XL and Challenge 2017 (download) |
| H | Optional: dynamic int8 quantization in PyTorch | synthetic beats |

All synthetic values (wave sizes, noise, labeller accuracies, registry distributions) are illustrative. Each
part ends with **checks**; the lab runs in under a minute (plus downloads in part G). Files are written to a
temporary folder.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import aiecg as ai
from epml import deploy as dp
from epml import foundation as fm
from epml import trust as tr
from epml.ml import auc_rank, cohen_kappa, measured_accuracy
from epml.retrieval import wilson_interval

WORK = Path(tempfile.mkdtemp(prefix="m47_lab_"))
print("working folder:", WORK)

## Part A · Units in file headers, and an audit that catches mistakes

A WFDB record stores integers; its header says how to turn them into physical units: $v = (d - b)/g$ with gain $g$
in units per mV and baseline $b$ (eq-m47-adu; PhysioNet WFDB header format). You write 30 synthetic recordings at
250 samples per second with gain 200, then export them three ways: correctly, with a header that claims gain 1000,
and with a header that claims 500 samples per second. The signal files are identical; only the headers differ.

The audit knows nothing about physiology. It learns the 1st to 99th percentile of the median R amplitude and the
heart rate from 200 training recordings, and flags a new recording outside either range.

In [ ]:
FS_A = 250.0
r = np.random.default_rng(1)

def ecg(r):
    rhythm = "af" if r.random() < 0.2 else "sinus"
    return ai.synth_ecg(rhythm, FS_A, 10.0, r.uniform(60, 100), r.uniform(0.6, 1.4), rng=r)[0]

train = [ecg(r) for _ in range(200)]
audit = dp.Audit(train, FS_A)
print(f"Training ranges: R amplitude {audit.amp[0]:.2f}-{audit.amp[1]:.2f} mV, heart rate {audit.hr[0]:.0f}-{audit.hr[1]:.0f} per minute")

new = [ecg(r) for _ in range(30)]
exports = {"correct": {}, "gain 1000 in header": {"gain": 1000}, "fs 500 in header": {"fs": 500}}
flags, worst = {}, 0.0
for kind, edit in exports.items():
    folder = WORK / "A" / kind.replace(" ", "_")
    folder.mkdir(parents=True, exist_ok=True)
    amp_bad = hr_bad = 0
    for i, x in enumerate(new):
        hea = dp.write_wfdb(x, FS_A, 200, folder, name=f"r{i:02d}")
        if edit:
            dp.edit_header(hea, **edit)
        y, fs, gain, units = dp.read_wfdb(hea)
        if kind == "correct":
            worst = max(worst, float(np.max(np.abs(y - x))))
        c = audit.check(y, fs)
        amp_bad += not c["amp_ok"]; hr_bad += not c["hr_ok"]
    flags[kind] = (amp_bad / len(new), hr_bad / len(new))
    print(f"{kind:22s} flagged for amplitude {flags[kind][0]:.0%}, for heart rate {flags[kind][1]:.0%}")
print(f"Largest round-trip error with the correct header: {1000 * worst:.2f} µV (bound 1/(2g) = {1000 / 400:.2f} µV)")

In [ ]:
# Checks
assert worst <= 1 / 400 + 1e-9
assert max(flags["correct"]) <= 0.1
assert flags["gain 1000 in header"][0] == 1.0 and flags["fs 500 in header"][1] == 1.0

Both mistakes give files that open without an error. A wrong gain shrinks every amplitude five-fold; a wrong
sampling rate doubles every heart rate. A model whose features are ratios of RR intervals would not notice the
second one at all, which is why the check belongs in the data pipeline, not in the model.

## Part B · Checksums, a manifest and a dataset version

A SHA-256 checksum changes if a single bit of a file changes. A manifest lists the checksum of every file; the
checksum of the manifest itself identifies the whole dataset version. This course pins PhysioNet's published
checksum lists the same way (`data/registry.json`).

In [ ]:
folder = WORK / "A" / "correct"
m1 = dp.manifest(folder)
v1 = dp.manifest_id(m1)
print(f"{len(m1)} files; dataset version {v1[:16]}…")

# Someone re-exports one record with a slightly different sample: same file name, same size.
dat = folder / "r07.dat"
raw = bytearray(dat.read_bytes()); raw[100] ^= 1; dat.write_bytes(bytes(raw))
m2 = dp.manifest(folder)
v2 = dp.manifest_id(m2)
diff = dp.diff_manifest(m1, m2)
print("changed:", diff["changed"], "| added:", diff["added"], "| removed:", diff["removed"])
print(f"new version {v2[:16]}…")

In [ ]:
# Checks
assert diff == {"changed": ["r07.dat"], "added": [], "removed": []} and v1 != v2

Record the version id next to every result: a number without the data version it came from cannot be
reproduced.

## Part C · Several labellers, majority votes and agreement

Each of $m$ labellers is right with probability $p$, independently; the majority of an odd number is right with the
probability in eq-m47-majority. Real labellers share hard cases: here 20% of recordings are hard, and on those each
labeller is right only 55% of the time (85% on the rest, so 79% on average). Agreement between two labellers is
Cohen's kappa (eq-m35-kappa), and the accuracy you can measure against imperfect labels is capped
(eq-m35-label-ceiling).

In [ ]:
r = np.random.default_rng(3)
y = (r.random(20_000) < 0.3).astype(int)
rows = []
for m in (1, 3, 5, 7):
    L_ind, _ = dp.simulate_labels(y, m, 0.79, r)
    L_cor, hard = dp.simulate_labels(y, m, 0.85, r, hard_share=0.2, p_hard=0.55)
    rows.append((m, dp.majority_accuracy(0.79, m), np.mean(dp.majority_vote(L_ind) == y),
                 np.mean(dp.majority_vote(L_cor) == y)))
print(" m  formula  independent  shared hard cases")
for m, f, a, b in rows:
    print(f"{m:2d}   {f:.3f}      {a:.3f}        {b:.3f}")
kappa = cohen_kappa(L_cor[:, 0], L_cor[:, 1])
true_acc = 0.95
print(f"Kappa between two labellers (shared hard cases): {kappa:.2f}")
print(f"A model with true accuracy {true_acc} measured against one labeller: {measured_accuracy(true_acc, 1 - 0.79):.3f}; "
      f"against a 7-labeller majority: {measured_accuracy(true_acc, 1 - rows[-1][3]):.3f}")

In [ ]:
# Checks
for m, f, a, b in rows:
    assert abs(a - f) < 0.01                        # eq-m47-majority holds for independent labellers
assert rows[-1][3] < rows[-1][2] - 0.03             # shared hard cases limit what a vote can fix
assert 0.2 < kappa < 0.8

The vote fixes independent mistakes, not shared ones: with hard cases, seven labellers barely beat three.
Adjudicating disagreements, or labelling hard cases with more information (a longer recording, a second lead),
helps where adding votes does not.

## Part D · Re-identification risk in a synthetic registry

A synthetic registry of 5,000 procedures (no real people) holds age, a sex code, a region code, the procedure day
and the device model. Direct identifiers are removed first. The remaining fields are quasi-identifiers: someone who
knows a person's values, and knows the person is in the registry, finds the records that match. With $f_i$ matching
records the chance of picking the right one is $1/f_i$ (eq-m47-risk). You coarsen fields, and finally group the
rare oldest and youngest ages, until every class has at least $k = 5$ records; the target and the age cut-offs are
illustrative, not a legal standard.

In [ ]:
reg = dp.synthetic_registry(5000, np.random.default_rng(4))
plans = [
    ("everything, age in years, exact day", dict(age_width=1, date="day"), ["age", "sex", "device", "region", "date"]),
    ("month instead of day", dict(age_width=1, date="month"), ["age", "sex", "device", "region", "date"]),
    ("no date", dict(age_width=1, date="none"), ["age", "sex", "device", "region"]),
    ("no date, 5-year age groups", dict(age_width=5, date="none"), ["age", "sex", "device", "region"]),
    ("no date, no region, 5-year groups", dict(age_width=5, date="none", keep_region=False), ["age", "sex", "device"]),
    ("no date, no region, 10-year groups", dict(age_width=10, date="none", keep_region=False), ["age", "sex", "device"]),
    ("… and ages under 40 or over 89 grouped", dict(age_width=10, date="none", keep_region=False, age_bounds=(40, 89)),
     ["age", "sex", "device"]),
]
print(f"{'release':40s} {'unique':>7s} {'k':>4s} {'max risk':>9s} {'avg risk':>9s} {'classes':>8s}")
risks = []
for name, kw, cols in plans:
    t = dp.generalize(reg, **kw)
    f = dp.class_sizes(t, cols)
    rk = dp.reid_risk(f)
    n_classes = len(set(zip(*(t[c] for c in cols))))
    risks.append((name, rk, n_classes))
    print(f"{name:40s} {rk['unique']:7.1%} {rk['k']:4d} {rk['max']:9.2f} {rk['average']:9.3f} {n_classes:8d}")
first_ok = next((name for name, rk, _ in risks if rk["k"] >= 5), None)
print("First release with every class of at least 5 records:", first_ok)

In [ ]:
# Checks
for name, rk, n_classes in risks:
    assert abs(rk["average"] - n_classes / 5000) < 1e-12    # mean of 1/f equals classes / records
assert risks[0][1]["unique"] > 0.9 and first_ok is not None

Each step trades detail for privacy: the release that reaches $k = 5$ has lost the date, the region and most of
the age, and the last unique records were the rare extremes of age, which no amount of coarsening in the middle of
the range fixes. Table de-identification is also not the whole story for ECG data: deep networks can identify people
from their ECGs (Ghazarian et al. 2021), so releasing waveforms needs its own risk assessment and usually a data-use
agreement.

## Part E · Quantizing inputs and weights

First the input: a beat stored with $b$ bits over its own range or over a fixed ±5 mV range (as in m47-d1). Then
the model: the small network of m46 (wide-QRS beats) with its weights stored as 8-bit or 4-bit integers, one scale
per tensor or one per hidden unit (eq-m47-affine with zero point 0). Rounding error is uniform with variance
$S^2/12$ (eq-m47-qnoise).

In [ ]:
beat = fm.beat_shape(fm.BEAT_T)
print("bits  own range: SQNR (pred)   ±5 mV range: SQNR (pred)")
sq = {}
for b in (6, 8, 10, 12):
    own, S_own = dp.fake_quant(beat, b)
    wide, S_wide = dp.fake_quant(beat, b, rmin=-5, rmax=5)
    sq[b] = (dp.sqnr_db(beat, own), dp.predicted_sqnr_db(beat, S_own), dp.sqnr_db(beat, wide), dp.predicted_sqnr_db(beat, S_wide))
    print(f"{b:4d}   {sq[b][0]:6.1f} ({sq[b][1]:5.1f})        {sq[b][2]:6.1f} ({sq[b][3]:5.1f})")

rng = np.random.default_rng(5)
P = fm.patient_params(1500, rng)
X = fm.recordings(P, np.arange(1500), rng)
y_wide = (P["width"] > 1.4).astype(int)
mu, sd = X.mean(axis=0), X.std(axis=0) + 1e-9
Zx = (X - mu) / sd
mlp = tr.TinyMLP(Zx.shape[1], 16, seed=0).fit(Zx[:1000], y_wide[:1000], lr=0.5, epochs=600)
p32 = mlp.prob(Zx[1000:])
results = {}
for bits in (8, 4):
    for per_channel in (False, True):
        q = dp.quantize_mlp(mlp, bits, per_channel)
        pq = q.prob(Zx[1000:])
        results[(bits, per_channel)] = (auc_rank(pq, y_wide[1000:]), np.mean(np.abs(pq - p32)),
                                        np.mean((pq >= 0.5) != (p32 >= 0.5)), dp.model_bytes(mlp, bits))
print(f"float32: AUC {auc_rank(p32, y_wide[1000:]):.3f}, {dp.model_bytes(mlp, 32)} bytes")
for (bits, pc), (auc, dp_mean, flips, nbytes) in results.items():
    print(f"int{bits} {'per hidden unit' if pc else 'per tensor     '}: AUC {auc:.3f}, mean |change in p| {dp_mean:.4f}, "
          f"decisions changed {flips:.1%}, {nbytes} bytes")

S_w = np.max(np.abs(mlp.W)) / 127
err_w = dp.quantize_mlp(mlp, 8).W - mlp.W
print(f"Weight rounding error variance {err_w.var():.3e}; S^2/12 = {S_w**2 / 12:.3e}")

In [ ]:
# Checks
assert abs(sq[8][0] - sq[8][1]) < 3 and sq[8][0] > sq[8][2] + 10   # wasted range costs SQNR
assert results[(8, False)][1] < results[(4, False)][1]
assert results[(4, True)][1] <= results[(4, False)][1]
assert abs(results[(8, False)][0] - auc_rank(p32, y_wide[1000:])) < 0.01
assert 0.5 < err_w.var() / (S_w**2 / 12) < 1.5

Eight bits barely change this model; four bits do, and one scale per hidden unit helps because the rows of
the weight matrix have different ranges. Always compare the quantized model with the original on the same test
data, including subgroups, before deployment: quantization is a model change (m45).

## Part F · A model card from the lab's own results

The card follows the idea of Mitchell et al. (2019): intended use, out-of-scope uses, data, metrics overall and by
group, limitations, and here also the data version and the checksum of the deployed weights. Every number is
filled in from parts B and E; `dp.model_card` refuses a card with an empty section.

In [ ]:
import json

weights = WORK / "F" / "wide_qrs_int8.json"
weights.parent.mkdir(parents=True, exist_ok=True)
q8 = dp.quantize_mlp(mlp, 8, per_channel=True)
weights.write_text(json.dumps({"W": q8.W.tolist(), "b": q8.b.tolist(), "v": q8.v.tolist(), "c": q8.c}))
pq8 = q8.prob(Zx[1000:])
yt = y_wide[1000:]
def sens_spec(p, y):
    tp, fn = int(np.sum((p >= 0.5) & (y == 1))), int(np.sum((p < 0.5) & (y == 1)))
    tn, fp = int(np.sum((p < 0.5) & (y == 0))), int(np.sum((p >= 0.5) & (y == 0)))
    return tp, tp + fn, tn, tn + fp
groups = {"low amplitude (scale < 1)": P["scale"][1000:] < 1, "high amplitude (scale >= 1)": P["scale"][1000:] >= 1}
by_group = []
for name, m in groups.items():
    tp, npos, tn, nneg = sens_spec(pq8[m], yt[m])
    lo, hi = wilson_interval(tp, npos)
    by_group.append(f"| {name} | {int(m.sum())} | {tp / npos:.2f} ({lo:.2f}–{hi:.2f}) | {tn / nneg:.2f} |")
tp, npos, tn, nneg = sens_spec(pq8, yt)
card = dp.model_card({
    "Model details": "Wide-QRS beat classifier: a 200-16-1 tanh network (m46), weights stored as int8 with one scale "
                     "per hidden unit. Course example, version 0.1.",
    "Intended use": "Teaching: shows how a model card is generated from results. Synthetic beats only.",
    "Out of scope": "Any real patient, any clinical decision, any ECG device. Not validated on real recordings.",
    "Training data": "1,000 synthetic beats (m43 beat model), wide QRS = width factor above 1.4; values illustrative.",
    "Evaluation data": "500 further synthetic beats from the same generator (no external validation).",
    "Metrics": f"AUC {auc_rank(pq8, yt):.3f} (float32 model {auc_rank(p32, yt):.3f}); sensitivity {tp / npos:.2f}, "
               f"specificity {tn / nneg:.2f} at probability 0.5.",
    "Metrics by group": "| Group | n | Sensitivity (95% CI) | Specificity |\n| --- | --- | --- | --- |\n" + "\n".join(by_group),
    "Limitations": "Synthetic data from one generator; groups chosen for illustration; no prospective evaluation.",
    "Version and checksums": f"Data version (part B manifest) {v2}; weights file SHA-256 {dp.sha256_file(weights)}.",
})
(WORK / "F" / "model_card.md").write_text(card)
print(card)

In [ ]:
# Checks
assert dp.missing_sections(card) == [] and v2 in card and dp.sha256_file(weights) in card

## Part G · Real headers, label-quality fields and an audit across databases

PTB-XL 1.0.3 (Wagner et al. 2020; Creative Commons Attribution 4.0) ships each recording at two sampling rates;
the PhysioNet/CinC Challenge 2017 (Clifford et al. 2017; Open Data Commons Attribution v1.0) has single-lead
recordings from a different device. You read only the small header files, list what they declare, count
PTB-XL's own label-quality fields as the database codes them, and check how many PTB-XL recordings an audit built
on Challenge 2017 would flag. Without internet access this part is skipped.

In [ ]:
from collections import Counter
from scipy.signal import resample_poly

from epml.data import (DataUnavailable, challenge2017_headers, load_challenge2017, load_ptbxl_lead,
                       ptbxl_columns, ptbxl_headers)

def describe(h):
    return (float(h.fs), int(h.n_sig), tuple(sorted(set(h.units))), tuple(sorted(set(float(g) for g in h.adc_gain))),
            tuple(sorted(set(int(b) for b in h.baseline))), tuple(sorted(set(h.fmt))))

try:
    lr, hr = ptbxl_headers(8, rate="lr"), ptbxl_headers(8, rate="hr")
    ch = challenge2017_headers(8)
    cols, names = ptbxl_columns(("validated_by_human", "second_opinion", "device"), fold=10)
    sig, lab, _ = load_challenge2017(n_per_class=60, classes=("N", "A"))
    ext, ext_lab, _, _ = load_ptbxl_lead(n_per_class=40, codes=("SR", "AFIB"), fold=10)
    REAL = True
except DataUnavailable as err:
    print("Part G skipped:", err)
    REAL = False

if REAL:
    for name, heads in (("PTB-XL 100 Hz", lr), ("PTB-XL 500 Hz", hr), ("Challenge 2017", ch)):
        kinds = Counter(describe(h) for _, h in heads)
        print(name)
        for (fs, n_sig, units, gains, bases, fmts), k in kinds.items():
            print(f"  {k} headers: fs {fs:g} Hz, {n_sig} signals, units {units}, gains {gains}, baselines {bases}, formats {fmts}")
    print("PTB-XL fold 10 columns present:", [c for c in ("validated_by_human", "second_opinion", "device") if c in names])
    for field in ("validated_by_human", "second_opinion"):
        if field in names:
            print(f"  {field}: {Counter(r[field] for r in cols).most_common()}")
    if "device" in names:
        print("  most common device codes:", Counter(r["device"] for r in cols).most_common(4))

In [ ]:
if REAL:
    ch100 = [resample_poly(np.asarray(s, float), 1, 3) for s in sig]      # 300 Hz to 100 Hz
    audit_ch = dp.Audit(ch100, 100.0)
    checks = [audit_ch.check(np.asarray(x, float), 100.0) for x in ext]
    amp_flag = np.mean([not c["amp_ok"] for c in checks]); hr_flag = np.mean([not c["hr_ok"] for c in checks])
    print(f"Audit built on Challenge 2017: R amplitude {audit_ch.amp[0]:.2f}-{audit_ch.amp[1]:.2f}, "
          f"heart rate {audit_ch.hr[0]:.0f}-{audit_ch.hr[1]:.0f}")
    print(f"PTB-XL lead I recordings flagged: amplitude {amp_flag:.0%}, heart rate {hr_flag:.0%} (of {len(ext)})")

In [ ]:
# Checks
if REAL:
    fs_lr = {float(h.fs) for _, h in lr}; fs_hr = {float(h.fs) for _, h in hr}
    assert len(fs_lr) == 1 and len(fs_hr) == 1 and max(fs_hr) > max(fs_lr)
    assert 0 <= amp_flag <= 1 and 0 <= hr_flag <= 1

Read the headers before the signals: they tell you the sampling rate, units and gain that every later number
depends on. Recordings from another database can look unusual to an audit even when nothing is wrong with them; a
flag means "look", not "reject".

## Part H · Optional: dynamic int8 quantization in PyTorch

PyTorch's dynamic quantization stores the weights of linear layers as int8 and quantizes activations on the fly.
Runs only if PyTorch is installed and its CPU build has a quantized engine.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part H skipped.")

gap_q = None
if torch is not None:
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Linear(Zx.shape[1], 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    Xt = torch.tensor(Zx[:1000], dtype=torch.float32); yt_t = torch.tensor(y_wide[:1000], dtype=torch.float32)
    for _ in range(300):
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(net(Xt).squeeze(1), yt_t)
        loss.backward(); opt.step()
    engines = torch.backends.quantized.supported_engines
    engine = next((e for e in ("fbgemm", "x86", "qnnpack") if e in engines), None)
    if engine is None:
        print("No quantized engine in this PyTorch build: part H skipped.")
    else:
        torch.backends.quantized.engine = engine
        qnet = torch.ao.quantization.quantize_dynamic(net, {torch.nn.Linear}, dtype=torch.qint8)
        with torch.no_grad():
            Xv = torch.tensor(Zx[1000:], dtype=torch.float32)
            p_f, p_q = torch.sigmoid(net(Xv)).squeeze(1).numpy(), torch.sigmoid(qnet(Xv)).squeeze(1).numpy()
        gap_q = float(np.mean(np.abs(p_f - p_q)))
        print(f"Engine {engine}: AUC float {auc_rank(p_f, y_wide[1000:]):.3f}, int8 {auc_rank(p_q, y_wide[1000:]):.3f}; "
              f"mean |change in p| {gap_q:.4f}")

In [ ]:
# Checks
if gap_q is not None:
    assert gap_q < 0.02

## Where to go next

- In part A, add a third mistake: a header that says µV while the data are in mV. What does the audit do?
- In part C, make the hard cases 40% of recordings. When does a 7-labeller majority stop helping?
- In part D, keep the region but drop the device. Which field carries more risk in this registry?
- In part E, quantize the hidden activations too, choosing their range from a calibration set. Compare the
  minimum-to-maximum range with the 1st to 99th percentile.